In [1]:
import numpy as np
import matplotlib.pyplot as plt

def fit_tls(x, y):
    """Fits a line ax + by + c = 0 using Total Least Squares."""
    x_mean, y_mean = np.mean(x), np.mean(y)
    # Center the data
    u = np.vstack([x - x_mean, y - y_mean]).T
    # SVD
    _, _, vh = np.linalg.svd(u)
    # The normal vector (a, b) is the last row of vh
    a, b = vh[-1]
    c = -(a * x_mean + b * y_mean)
    return a, b, c

def run_ransac(x, y, iterations=100, threshold=0.1):
    """Finds the best line in a noisy dataset using RANSAC."""
    best_inliers = []
    best_params = (0, 0, 0)
    
    n_points = len(x)
    for _ in range(iterations):
        # Sample 2 random points
        idx = np.random.choice(n_points, 2, replace=False)
        p1, p2 = np.array([x[idx], y[idx]]).T
        
        # Define line through p1 and p2: (y2-y1)x - (x2-x1)y + x2y1 - y2x1 = 0
        a = p2[1] - p1[1]
        b = p1[0] - p2[0]
        c = p2[0] * p1[1] - p2[1] * p1[0]
        
        # Normalize
        norm = np.sqrt(a**2 + b**2)
        if norm == 0: continue
        a, b, c = a/norm, b/norm, c/norm
        
        # Calculate distances
        distances = np.abs(a*x + b*y + c)
        inliers = np.where(distances < threshold)[0]
        
        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_params = (a, b, c)
            
    return best_params, best_inliers

# Load Data
data = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)
# Part (a)
a_tls, b_tls, c_tls = fit_tls(data[:, 0], data[:, 3])
print(f"Task 1(a) TLS Parameters (ax + by + c = 0): {a_tls:.4f}x + {b_tls:.4f}y + {c_tls:.4f} = 0")

# Part (b)
X_all = data[:, :3].flatten()
Y_all = data[:, 3:].flatten()

remaining_x, remaining_y = X_all.copy(), Y_all.copy()
print("\nTask 1(b) Iterative RANSAC results:")
for i in range(3):
    params, inlier_indices = run_ransac(remaining_x, remaining_y, threshold=0.05)
    print(f"Line {i+1} params: {params}")
    # Remove inliers
    remaining_x = np.delete(remaining_x, inlier_indices)
    remaining_y = np.delete(remaining_y, inlier_indices)

Task 1(a) TLS Parameters (ax + by + c = 0): 0.7736x + -0.6337y + -3.7942 = 0

Task 1(b) Iterative RANSAC results:
Line 1 params: (np.float64(0.3464109964507718), np.float64(0.9380828436433444), np.float64(-1.9630760027068883))
Line 2 params: (np.float64(-0.7229050732024149), np.float64(0.6909473606131014), np.float64(-0.650579924216045))
Line 3 params: (np.float64(0.7517873997688184), np.float64(-0.6594055698497236), np.float64(1.0797753003337183))
